In [0]:
# Import Libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
# Create Spark Session
spark = SparkSession.builder \
    .appName("Week 5 Spark Assignment") \
    .getOrCreate()

print("Spark Version:", spark.version)

Spark Version: 4.1.0


In [0]:
# Load Dataset
df = spark.table("default.dataset")

In [0]:
# 5: Display Dataset
df.show(5, truncate=False)

+----------+--------------+------------------------------------------+------+-------------+-------------+--------+-----------+--------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [0]:
# Check Total Rows
print("Total Rows:", df.count())

Total Rows: 1000


In [0]:
# Check Total Columns
print("Total Columns:", len(df.columns))

Total Columns: 24


In [0]:
# Print Schema
df.printSchema()

root
 |-- product_id: long (nullable = true)
 |-- title: string (nullable = true)
 |-- product_description: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- ratings_count: long (nullable = true)
 |-- initial_price: long (nullable = true)
 |-- discount: long (nullable = true)
 |-- final_price: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- images: string (nullable = true)
 |-- delivery_options: string (nullable = true)
 |-- product_details: string (nullable = true)
 |-- breadcrumbs: string (nullable = true)
 |-- product_specifications: string (nullable = true)
 |-- amount_of_stars: string (nullable = true)
 |-- what_customers_said: string (nullable = true)
 |-- seller_name: string (nullable = true)
 |-- sizes: string (nullable = true)
 |-- videos: string (nullable = true)
 |-- seller_information: string (nullable = true)
 |-- variations: string (nullable = true)
 |-- best_offer: string (nullable = true)
 |-- more_offers: string (nullable = true)

In [0]:
# Print Column Names
print(df.columns)

['product_id', 'title', 'product_description', 'rating', 'ratings_count', 'initial_price', 'discount', 'final_price', 'currency', 'images', 'delivery_options', 'product_details', 'breadcrumbs', 'product_specifications', 'amount_of_stars', 'what_customers_said', 'seller_name', 'sizes', 'videos', 'seller_information', 'variations', 'best_offer', 'more_offers', 'category']


In [0]:
df.describe().show()

+-------+------------------+------------------+--------------------+------------------+------------------+------------------+------------------+-----------+--------+--------------------+--------------------+--------------------+--------------------+----------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+---------+
|summary|        product_id|             title| product_description|            rating|     ratings_count|     initial_price|          discount|final_price|currency|              images|    delivery_options|     product_details|         breadcrumbs|product_specifications|     amount_of_stars| what_customers_said|         seller_name|               sizes|              videos|  seller_information|          variations|          best_offer|         more_offers| category|
+-------+------------------+------------------+---------

Q1: What are the key limitations of traditional MapReduce that make Spark a preferred choice for modern big data processing?

MapReduce is a disk-based distributed processing framework where intermediate results are written to disk after each processing stage. This causes significant disk I/O, making iterative and interactive workloads slow.
Apache Spark addresses these limitations by processing data in memory whenever possible, reducing disk access and improving performance.

| Feature              | MapReduce              | Apache Spark                  |
| -------------------- | ---------------------- | ----------------------------- |
| Processing           | Disk-based             | In-memory                     |
| Speed                | Slower                 | Much Faster                   |
| Iterative Processing | Inefficient            | Efficient                     |
| Real-time Processing | Not suitable           | Supported                     |
| Ease of Development  | Complex                | Easy with DataFrames and APIs |
| Machine Learning     | Limited                | Built-in MLlib                |
| Streaming            | Not supported natively | Supported                     |

    Limitations of MapReduce
Writes intermediate data to disk after every stage.
High disk I/O overhead.
Slow for iterative algorithms.
Higher latency.
More boilerplate code.
No native support for streaming or interactive analytics.

    Spark is Preferred
In-memory computation.
Faster execution.
Easy-to-use DataFrame API.
Supports SQL, Streaming, Machine Learning, and Graph Processing.
Better fault tolerance through lineage.

Q2: Explain how Spark uses In-Memory Computing to speed up iterative machine learning algorithms compared to disk-based systems.

Spark stores intermediate computation results in RAM instead of repeatedly writing them to disk. This allows iterative algorithms, such as machine learning and graph processing, to reuse data directly from memory, significantly reducing execution time.
In contrast, MapReduce reads and writes data to disk between every iteration, increasing latency.

Example

    MapReduce
Read Data
   ↓
Process
   ↓
Write to Disk
   ↓
Read Again
   ↓
Process Again

    Spark
Read Data
   ↓
Load into Memory
   ↓
Iteration 1
   ↓
Iteration 2
   ↓
Iteration 3

    Benefits
Faster execution.
Reduced disk I/O.
Efficient caching.
Ideal for iterative machine learning.
Better interactive analytics.

In [0]:
# Q3 - Remove Duplicate Rows
# The assignment mentions user_id and transaction_date, but your dataset doesn't have those columns.
# We'll remove duplicates using the unique product identifier.
# The dataset does not contain user_id or transaction_date. Therefore, product_id is used as the unique identifier to remove duplicate products.

df_unique = df.dropDuplicates(["product_id"])
print("Original Rows :", df.count())
print("Rows After Removing Duplicates :", df_unique.count())
df_unique.show(5)

Original Rows : 1000
Rows After Removing Duplicates : 1000
+----------+--------------+--------------------+------+-------------+-------------+--------+-----------+--------+--------------------+--------------------+--------------------+--------------------+----------------------+--------------------+-------------------+-----------+--------------------+--------------------+------------------+--------------------+----------+--------------------+---------+
|product_id|         title| product_description|rating|ratings_count|initial_price|discount|final_price|currency|              images|    delivery_options|     product_details|         breadcrumbs|product_specifications|     amount_of_stars|what_customers_said|seller_name|               sizes|              videos|seller_information|          variations|best_offer|         more_offers| category|
+----------+--------------+--------------------+------+-------------+-------------+--------+-----------+--------+--------------------+-----------

In [0]:
# Q4. Given a DataFrame df_sales, filter for rows where the region is 'West' and group by product_category to find the average sale_amount.

# The dataset does not contain a **Region** column. Therefore, products are grouped by **Category** and the average **Final Price** is calculated.

from pyspark.sql.functions import avg, regexp_replace
avg_price = (
    df.groupBy("category")
      .agg(avg(regexp_replace(regexp_replace(regexp_replace("final_price", '"₹', ''), ',', ''), '"', '').cast("double")).alias("Average_Final_Price"))
)
avg_price.show(truncate=False)

+--------------------+-------------------+
|category            |Average_Final_Price|
+--------------------+-------------------+
|backpacks           |2892.0             |
|bath-robe           |2899.0             |
|bathroom-accessories|1399.0             |
|bath-towels         |1399.0             |
|bedsheets           |2444.5454545454545 |
|belts               |223.0              |
|bodysuit            |1249.0             |
|boots               |2463.0             |
|boxers              |367.0              |
|bra                 |794.7692307692307  |
|bracelet            |750.75             |
|briefs              |395.9166666666667  |
|camisoles           |247.5              |
|candle-holders      |714.0              |
|caps                |961.0              |
|casual-shoes        |2660.782608695652  |
|chair-cover         |935.0              |
|clothing-set        |516.5              |
|clutches            |1980.0             |
|co-ords             |1700.0             |
+----------

In [0]:
# Q5. Difference Between .na.drop() and .na.fill()

# .na.drop()
# - Removes rows containing null values.

# .na.fill()
# - Replaces null values with a specified value without removing rows.
# In this dataset, missing values in the **delivery_options** column are replaced with "Unknown".

clean_df = df.na.fill({"delivery_options": "Unknown"})
clean_df.select("delivery_options").show(10, truncate=False)

+-----------------------------------------------------------------------------------------------------------------------------------+
|delivery_options                                                                                                                   |
+-----------------------------------------------------------------------------------------------------------------------------------+
|["100% Original Products","Pay on delivery might be available","Easy 14 days returns and exchanges","Try & Buy might be available"]|
|["100% Original Products","Pay on delivery might be available","Easy 14 days returns and exchanges","Try & Buy might be available"]|
|["100% Original Products","Pay on delivery might be available","Easy 14 days returns and exchanges","Try & Buy might be available"]|
|["100% Original Products","Pay on delivery might be available","Easy 14 days returns and exchanges","Try & Buy might be available"]|
|["100% Original Products","Pay on delivery might be available

In [0]:
# Q6. Count Products by Category
# Since the dataset does not contain a city column, the records are grouped by category instead.

from pyspark.sql.functions import count
category_count = (
    df.groupBy("category")
      .agg(count("*").alias("Total_Products"))
      .filter(col("Total_Products") > 100)
)
category_count.show(truncate=False)

+--------+--------------+
|category|Total_Products|
+--------+--------------+
|tops    |122           |
+--------+--------------+



In [0]:
# Q7. DataFrame Immutability
# Spark DataFrames are immutable.
# Any operation such as:
# - drop()
# - filter()
# - withColumnRenamed()
# creates a new DataFrame instead of modifying the original DataFrame.

# Rename a column:
renamed_df = df.withColumnRenamed("title", "product_title")
renamed_df.show(5, truncate=False)

# Drop a column:
new_df = df.drop("videos")
new_df.printSchema()

+----------+--------------+------------------------------------------+------+-------------+-------------+--------+-----------+--------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [0]:
# Q8. Filter Dataset Using Multiple Conditions

# The assignment mentions filtering by **age** and **subscription**, but these columns are not available in the dataset.
# Instead, products are filtered where:
# - Rating is greater than or equal to 4
# - Final Price is greater than 500

from pyspark.sql.functions import regexp_replace
filtered_df = df.filter(
    (col("rating") >= 4) &
    (regexp_replace(regexp_replace(regexp_replace("final_price", '"₹', ''), ',', ''), '"', '').cast("double") > 500)
)
filtered_df.show(truncate=False)

+----------+---------------+----------------------------------------------------------------------------------+------+-------------+-------------+--------+-----------+--------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [0]:
# Q9. Why Handle Null Values Before Aggregation?

# Handling null values before applying aggregation functions such as sum() or avg() ensures accurate calculations and prevents missing values from affecting the results.
# Benefits:
# - Improves data quality
# - Produces accurate statistics
# - Prevents unexpected null outputs

from pyspark.sql.functions import regexp_replace
clean_df = df.na.fill({"final_price": 0})
clean_df.select(avg(regexp_replace(regexp_replace(regexp_replace("final_price", '"₹', ''), ',', ''), '"', '').cast("double")).alias("Average Price")).show()

+-------------+
|Average Price|
+-------------+
|     1706.096|
+-------------+



In [0]:
# Q10. Cast and Rename Column

# The dataset does not contain a timestamp column.
# For demonstration, the **final_price** column is cast to DoubleType and renamed as **price**.

from pyspark.sql.functions import regexp_replace
cast_df = (
    df.withColumn(
        "price",
        regexp_replace(regexp_replace(regexp_replace(col("final_price"), '"₹', ''), ',', ''), '"', '').cast(DoubleType())
    )
    .drop("final_price")
)
cast_df.show(5, truncate=False)

+----------+--------------+------------------------------------------+------+-------------+-------------+--------+--------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [0]:
# Q11. Shuffle Process

# A shuffle occurs when Spark redistributes data between partitions during operations such as:
# - groupBy()
# - join()
# - distinct()
# - orderBy()
# Shuffle is considered a wide transformation because data moves across partitions, making it more expensive than narrow transformations.



In [0]:
# Q12. Remove Null or Empty Values

# The assignment refers to email and username columns.
# This dataset does not contain those columns.
# Instead, rows with null product titles or empty seller names are removed.

clean_data = df.filter(
    col("title").isNotNull() &
    (trim(col("seller_name")) != "")
)
clean_data.show(5, truncate=False)

+----------+--------------+--------------------------------------------------------------+------+-------------+-------------+--------+-----------+--------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [0]:
## Q13. Multiple Aggregations
# The agg() function calculates multiple statistics simultaneously.
from pyspark.sql.functions import regexp_replace
df.agg(
    min(regexp_replace(regexp_replace(regexp_replace("final_price", '"₹', ''), ',', ''), '"', '').cast("double")).alias("Minimum Price"),
    max(regexp_replace(regexp_replace(regexp_replace("final_price", '"₹', ''), ',', ''), '"', '').cast("double")).alias("Maximum Price"),
    avg(regexp_replace(regexp_replace(regexp_replace("final_price", '"₹', ''), ',', ''), '"', '').cast("double")).alias("Average Price")
).show()

+-------------+-------------+-------------+
|Minimum Price|Maximum Price|Average Price|
+-------------+-------------+-------------+
|        199.0|      17995.0|     1706.096|
+-------------+-------------+-------------+



In [0]:
# Q14. Risk of inferSchema=True

# When source data contains inconsistent or messy values, inferSchema=True may incorrectly infer the data type.
# Possible issues:
# - Date columns may be read as strings.
# - Numeric values may become strings.
# - Inconsistent formats can cause parsing errors.
# It is good practice to verify the schema after loading the data.

In [0]:
## Q15. Final Data Processing Pipeline

# Pipeline Steps:
# 1. Remove duplicate products.
# 2. Fill missing final prices with 0.
# 3. Group by seller name.
# 4. Calculate total revenue.

from pyspark.sql.functions import regexp_replace
pipeline_df = (
    df.dropDuplicates(["product_id"])
      .na.fill({"final_price": 0})
      .groupBy("seller_name")
      .agg(sum(regexp_replace(regexp_replace(regexp_replace("final_price", '"₹', ''), ',', ''), '"', '').cast("double")).alias("Total_Revenue"))
)

pipeline_df.show(truncate=False)

+--------------------------------------------------+-------------+
|seller_name                                       |Total_Revenue|
+--------------------------------------------------+-------------+
|NULL                                              |685018.0     |
|H &███Hen███ & █████████eta█████████████████████  |7136.0       |
|Sig███Ove███as                                    |865.0        |
|Bel███Cas███ash█████████ail█████████              |404.0        |
|Kas███r L███her█████████Pvt██████                 |223.0        |
|PVH███VIN███ASH█████████TE ███████████████        |10375.0      |
|ETE███LIA███EAT█████████CHA█████████████████████ED|699.0        |
|MAX███ IN███NAT██████                             |1087.0       |
|VOL███FAS███N P█████████                          |3839.0       |
|THE███ULE███TOR█████████D.                        |367.0        |
|Tru███m R███il                                    |21246.0      |
|Gro███son███ppa█████████Ltd███                    |299.0     

In [0]:
pipeline_df.write.mode("overwrite").saveAsTable("default.pipeline_results")

In [0]:
%sql
SELECT * FROM default.pipeline_results

seller_name,Total_Revenue
PRE███TI ███LEC██████,1995.0
AAR███LIF███YLE█████████LIM██████,14512.0
Blo███Exi███riv█████████ed,854.0
Gun███al ███tch█████████E L█████████,2100.0
Home,1110.0
ICO███ FA███ON █████████ PV██████,6748.0
CRO███OAD███LOT█████████LTD███,2327.0
Kis███App███ls █████████imi██████,28823.0
Kri███n A███rel█████████Lim██████,1102.0
STR███LIN███PPA█████████ATE█████████,1584.0
